In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression
import os

In [2]:
# Cargamos los csv de los tifs
path = "saved_files/dataset"
dfs = {}
for archivo in os.listdir(path):
    if archivo.endswith("_features.csv"):
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        dfs[nombre_sin_extension[:-9]] = pd.read_csv(ruta_completa)

In [5]:
dfs_to_keep = [
    "C2X_3x3_merge_depth_lt_1", "C2X_5x5_merge_depth_lt_1", "C2X-Complex_5x5_merge_depth_lt_1", "C2X-Complex_3x3_merge_depth_lt_1",
    "C2X_3x3_merge_depth_gt_1", "C2X_5x5_merge_depth_gt_1", "C2X-Complex_5x5_merge_depth_gt_1", "C2X-Complex_3x3_merge_depth_gt_1",
    #"C2RCC_3x3_merge_depth_lt_1", "C2RCC_5x5_merge_depth_gt_1", "C2RCC_5x5_merge_depth_lt_1", "C2RCC_3x3_merge_depth_gt_1",
]

In [6]:
dfs = {k: dfs[k] for k in dfs_to_keep if k in dfs}

for nombre_df, df in dfs.items():
    for band_set in ["rhow", "rhown"]:
        dfs[nombre_df] = df.dropna()

In [23]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Invierno'
    elif month in [3, 4, 5]:
        return 'Primavera'
    elif month in [6, 7, 8]:
        return 'Verano'
    else:
        return 'Otoño'
    

def get_zone(buoy):
    if buoy in ["CTD1", "CTD2", "CTD3", "CTD4"]:
        return 'Zona-1'
    elif buoy in ["CTD6", "CTD8", "CTD9", "CTD10", "CTD12"]:
        return 'Zona-2'
    elif buoy in ["CTD7"]:
        return 'Zona-3'
    elif buoy in ["CTD11"]:
        return 'Zona-4'

In [24]:
for nombre_df, df in dfs.items():
    df["High_Chl"] = df["Chl"]>5
    df['Date'] = pd.to_datetime(df['Date'])
    df['Season'] = df['Date'].dt.month.apply(get_season)
    df['Zone'] = df['Buoy'].apply(get_zone)
    dfs[nombre_df] = df
    

In [39]:
df = dfs["C2X_3x3_merge_depth_lt_1"].iloc[:,4:]

In [48]:
# 1. Separar columnas numéricas y no numéricas
df_numericas = df.select_dtypes(include='number')
df_no_numericas = df.select_dtypes(exclude='number')

# 2. Calcular la correlación con 'Chl' solo entre columnas numéricas
correlaciones = df_numericas.corr()['Chl'].drop('Chl')

# 3. Filtrar predictores numéricos con correlación significativa
umbral_corr = 0.1
columnas_utiles = correlaciones[correlaciones.abs() >= umbral_corr].index.tolist()

# 4. Reconstruir el DataFrame con:
# - Las columnas numéricas útiles
# - La columna objetivo 'Chl'
df_filtrado = pd.concat([df[columnas_utiles + ['Chl']]], axis=1)


# Calcular la matriz de correlación entre predictores
corr_matrix = df_filtrado.drop(columns='Chl').corr().abs()

# Seleccionar columnas a eliminar (altamente correlacionadas entre sí)
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
columnas_redundantes = [col for col in upper.columns if any(upper[col] > 0.98)]

# Eliminar redundantes
df_final = pd.concat([df_filtrado.drop(columns=columnas_redundantes), df_no_numericas], axis=1)


In [49]:
df_final

,rhow_B1,rhow_B2,rhow_B4,rhow_B5,Band_15,dif_norm_rhow_B2_B3,dif_inv_rhow_B2_B3,dif_norm_rhow_B2_B4,dif_inv_rhow_B2_B4,dif_norm_rhow_B3_B4,...,dif_rel_4bands_rhow_B3_B4_B5_B2,sum_norm_3bands_rhow_B2_B4_B3,sum_norm_3bands_rhow_B3_B5_B4,dall_gitelson_rhown_B3_B4_B5,dif_norm_4_bands_rhown_B4_B5_B2_B3,sum_norm_3bands_rhown_B3_B5_B4,Chl,High_Chl,Season,Zone
0,0.005731,0.014207,0.018639,0.017016,0.518293,-0.462,44.458,-0.135,16.734,0.348,...,0.872,0.622,0.972,-0.434,0.055,0.949,10.0900,True,Verano,Zona-1
1,0.006148,0.011972,0.011034,0.010585,0.467246,-0.332,41.658,0.041,-7.105,0.368,...,1.281,0.642,0.987,-0.475,0.034,0.966,11.2050,True,Verano,Zona-1
2,0.006279,0.012704,0.012148,0.011127,0.502829,-0.321,38.247,0.022,-3.603,0.341,...,1.158,0.664,0.972,-0.447,0.040,0.959,12.0750,True,Verano,Zona-1
3,0.003612,0.006785,0.009603,0.012901,0.639833,-0.395,83.520,-0.172,43.253,0.240,...,-0.271,0.730,1.131,-0.380,-0.063,1.053,8.8400,True,Verano,Zona-1
4,0.006594,0.012779,0.009153,0.007928,0.311716,-0.306,36.681,0.165,-31.005,0.449,...,2.008,0.595,0.963,-0.515,0.042,0.953,12.1500,True,Verano,Zona-2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
446,0.007363,0.014684,0.002652,0.001563,0.042125,-0.059,7.585,0.694,-308.987,0.723,...,6.125,0.555,0.943,-0.531,0.031,0.950,0.2805,False,Primavera,Zona-2
447,0.024935,0.037002,0.025217,0.024079,0.634092,-0.250,10.824,0.189,-12.630,0.420,...,1.797,0.630,0.987,-0.532,0.027,0.970,0.4625,False,Primavera,Zona-2
448,0.002900,0.007073,0.000978,0.000523,0.000099,-0.079,20.764,0.757,-881.255,0.789,...,8.404,0.524,0.951,-0.516,0.025,0.958,0.6735,False,Primavera,Zona-2
449,0.015693,0.023755,0.006434,0.004399,0.142305,-0.095,7.307,0.574,-113.338,0.634,...,4.283,0.575,0.942,-0.545,0.036,0.946,0.4145,False,Primavera,Zona-4


In [34]:
columnas_redundantes

['rhow_B6',
 'rhow_B7',
 'rhow_B8',
 'rhown_B5',
 'rhown_B6',
 'dif_inv_rhow_B3_B4',
 'dif_inv_rhow_B3_B5',
 'dall_gitelson_rhow_B3_B4_B2',
 'dall_gitelson_rhow_B3_B5_B4',
 'dif_rel_4bands_rhow_B2_B4_B5_B3',
 'dif_rel_4bands_rhow_B2_B5_B4_B3',
 'dif_rel_4bands_rhow_B3_B4_B5_B2',
 'dif_rel_4bands_rhow_B3_B5_B4_B2',
 'dif_norm_rhown_B2_B3',
 'dif_norm_rhown_B2_B4',
 'dif_inv_rhown_B2_B4',
 'dif_norm_rhown_B2_B5',
 'dif_inv_rhown_B2_B5',
 'dif_norm_rhown_B3_B4',
 'dif_inv_rhown_B3_B4',
 'dif_norm_rhown_B3_B5',
 'dif_inv_rhown_B3_B5',
 'dif_inv_rhown_B4_B5',
 'dall_gitelson_rhown_B2_B3_B4',
 'dall_gitelson_rhown_B2_B3_B5',
 'dall_gitelson_rhown_B2_B4_B3',
 'dall_gitelson_rhown_B2_B5_B3',
 'dall_gitelson_rhown_B2_B5_B4',
 'dall_gitelson_rhown_B3_B4_B2',
 'dall_gitelson_rhown_B3_B5_B2',
 'dall_gitelson_rhown_B3_B5_B4',
 'dall_gitelson_rhown_B4_B5_B2',
 'dall_gitelson_rhown_B4_B5_B3',
 'dif_norm_4_bands_rhown_B2_B4_B3_B5',
 'dif_norm_4_bands_rhown_B2_B5_B3_B4',
 'dif_norm_4_bands_rhown_B3_B4_

In [41]:
df_final.columns

Index(['rhow_B4', 'rhow_B5', 'Band_15', 'dif_norm_rhow_B2_B3',
       'dif_norm_rhow_B2_B4', 'dif_inv_rhow_B2_B4', 'dif_norm_rhow_B3_B4',
       'dif_norm_rhow_B3_B5', 'dif_norm_rhow_B4_B5',
       'dall_gitelson_rhow_B2_B3_B4', 'dall_gitelson_rhow_B2_B3_B5',
       'dall_gitelson_rhow_B2_B4_B3', 'dall_gitelson_rhow_B2_B5_B3',
       'dall_gitelson_rhow_B2_B5_B4', 'dall_gitelson_rhow_B3_B4_B2',
       'dall_gitelson_rhow_B4_B5_B3', 'dif_norm_4_bands_rhow_B2_B4_B3_B5',
       'dif_norm_4_bands_rhow_B3_B4_B2_B5',
       'dif_norm_4_bands_rhow_B4_B5_B2_B3', 'dif_rel_4bands_rhow_B2_B3_B5_B4',
       'dif_rel_4bands_rhow_B2_B4_B3_B5', 'dif_rel_4bands_rhow_B2_B5_B3_B4',
       'dif_rel_4bands_rhow_B3_B2_B4_B5', 'dif_rel_4bands_rhow_B3_B2_B5_B4',
       'dif_rel_4bands_rhow_B3_B4_B5_B2', 'sum_norm_3bands_rhow_B2_B4_B3',
       'sum_norm_3bands_rhow_B3_B5_B4', 'dall_gitelson_rhown_B3_B4_B5',
       'dif_norm_4_bands_rhown_B4_B5_B2_B3', 'sum_norm_3bands_rhown_B3_B5_B4',
       'High_Chl', 'Chl'

In [35]:
df_final.columns

Index(['rhow_B5', 'Band_15', 'dif_norm_rhow_B2_B3', 'dif_norm_rhow_B2_B4',
       'dif_inv_rhow_B2_B4', 'dif_norm_rhow_B2_B5', 'dif_inv_rhow_B2_B5',
       'dif_norm_rhow_B3_B4', 'dif_norm_rhow_B3_B5', 'dif_norm_rhow_B4_B5',
       'dif_inv_rhow_B4_B5', 'dall_gitelson_rhow_B2_B3_B4',
       'dall_gitelson_rhow_B2_B3_B5', 'dall_gitelson_rhow_B2_B4_B3',
       'dall_gitelson_rhow_B2_B4_B5', 'dall_gitelson_rhow_B2_B5_B3',
       'dall_gitelson_rhow_B2_B5_B4', 'dall_gitelson_rhow_B3_B5_B2',
       'dall_gitelson_rhow_B4_B5_B2', 'dall_gitelson_rhow_B4_B5_B3',
       'dif_norm_4_bands_rhow_B2_B4_B3_B5',
       'dif_norm_4_bands_rhow_B2_B5_B3_B4',
       'dif_norm_4_bands_rhow_B3_B4_B2_B5',
       'dif_norm_4_bands_rhow_B3_B5_B2_B4', 'dif_rel_4bands_rhow_B2_B3_B5_B4',
       'dif_rel_4bands_rhow_B2_B4_B3_B5', 'dif_rel_4bands_rhow_B2_B5_B3_B4',
       'dif_rel_4bands_rhow_B3_B2_B4_B5', 'dif_rel_4bands_rhow_B4_B2_B5_B3',
       'dif_rel_4bands_rhow_B4_B3_B5_B2', 'sum_norm_3bands_rhow_B2_B4_B3',